In [2]:
import pandas as pd
import numpy as np

# ==============================================================================
# 1. DATA LOADING & PREPROCESSING
# ==============================================================================
df = pd.read_csv("/content/rahul_transactions.csv")

# Clean Date with dayfirst=True for mixed Indian banking formats
df['Date'] = pd.to_datetime(df['Date'], format='mixed', dayfirst=True)

# Clean Amount column (strip currency symbols and commas)
df['Amount'] = (
    df['Amount']
    .astype(str)
    .str.replace('₹', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.replace('Rs.', '', regex=False)
    .str.strip()
    .astype(float)
)

# Standardize Credit/Debit types
df['Type'] = df['Type'].str.upper().str.strip()
df['Type'] = df['Type'].apply(lambda x: 'Debit' if x in ['DEBIT', 'DR'] else 'Credit' if x in ['CREDIT', 'CR'] else x)

# ==============================================================================
# 2. VENDOR & CATEGORY EXTRACTOR (Strict String Matching - No Regex)
# ==============================================================================
def extract_vendor_and_category(desc):
    desc_upper = str(desc).upper()

    vendor = "Uncategorised"
    if any(k in desc_upper for k in ['SWIGGY', 'BUNDL TECH', 'INSTAMART']):
        vendor = 'Swiggy'
    elif 'ZOMATO' in desc_upper:
        vendor = 'Zomato'
    elif any(k in desc_upper for k in ['AMAZON', 'AMZN', 'FKART', 'FLIPKART']):
        vendor = 'Amazon' if 'AMAZON' in desc_upper or 'AMZN' in desc_upper else 'Flipkart'
    elif any(k in desc_upper for k in ['ZERODHA', 'GROWW', 'NEXTBILLION']):
        vendor = 'Zerodha' if 'ZERODHA' in desc_upper else 'Groww'
    elif 'ZEPTO' in desc_upper or 'KIRANAKART' in desc_upper:
        vendor = 'Zepto'
    elif 'BLINKIT' in desc_upper or 'GROFERS' in desc_upper:
        vendor = 'Blinkit'
    elif any(k in desc_upper for k in ['MYNTRA', 'NYKAA', 'FSN E-COMMERCE']):
        vendor = 'Myntra' if 'MYNTRA' in desc_upper else 'Nykaa'
    elif 'BOOKMYSHOW' in desc_upper or 'BIGTREE' in desc_upper:
        vendor = 'BookMyShow'
    elif any(k in desc_upper for k in ['UBER', 'RAPIDO', 'ROPPEN', 'OLA', 'BMTC', 'ANI TECHNOLOGIES']):
        vendor = 'Uber/Ola/Rapido'
    elif any(k in desc_upper for k in ['STARBUCKS', 'CAFE', 'THIRD WAVE', 'THIRDWAVE', 'TWC', 'COFFEE', 'CCD']):
        vendor = 'Cafes'
    elif any(k in desc_upper for k in ['MCDONALDS', 'KFC', 'BIRYANI', 'RESTAURANT', 'TRUFFLES', 'MEGHANA', 'DINEOUT']):
        vendor = 'Restaurants'
    elif 'SALARY' in desc_upper:
        vendor = 'Salary'
    elif any(k in desc_upper for k in ['BIGBASKET', 'DMART', 'AVENUE SUPERMARTS', 'INNOVATIVE RETAIL']):
        vendor = 'BigBasket / D-Mart'
    elif any(k in desc_upper for k in ['NETFLIX', 'SPOTIFY', 'HOTSTAR', 'STAR INDIA']):
        vendor = 'Subscriptions'
    elif any(k in desc_upper for k in ['BESCOM', 'BANGALORE ELEC', 'BWSSB', 'VI', 'VODAFONE', 'JIO', 'AIRTEL']):
        vendor = 'Utilities / Bills'
    elif 'RENT' in desc_upper:
        vendor = 'Rent / Housing'
    elif any(k in desc_upper for k in ['IOC', 'BPCL', 'HP PETROL', 'PETROL', 'INDIAN OIL']):
        vendor = 'Fuel'
    elif 'ATM-WDL' in desc_upper:
        vendor = 'ATM Cash Withdrawal'
    elif any(k in desc_upper for k in ['AMAN', 'ANKIT', 'VIKAS', 'PRIYA', 'NEHA', 'SNEHA', 'KARAN']):
        vendor = 'P2P UPI Transfers'

    cat = "Uncategorised"
    if vendor in ['Swiggy', 'Zomato']:
        cat = 'Food Delivery'
    elif vendor in ['Zepto', 'Blinkit', 'BigBasket / D-Mart']:
        cat = 'Quick Commerce'
    elif vendor in ['Amazon', 'Flipkart', 'Myntra', 'Nykaa']:
        cat = 'E-commerce'
    elif vendor in ['Zerodha', 'Groww']:
        cat = 'Investments'
    elif vendor in ['Cafes', 'Restaurants']:
        cat = 'Restaurants'
    elif vendor in ['BookMyShow', 'Subscriptions']:
        cat = 'Entertainment'
    elif vendor in ['Uber/Ola/Rapido', 'Fuel']:
        cat = 'Commute & Fuel'
    elif vendor in ['Utilities / Bills', 'Rent / Housing']:
        cat = 'Bills & Rent'
    elif vendor == 'P2P UPI Transfers':
        cat = 'P2P Transfers'
    elif vendor == 'ATM Cash Withdrawal':
        cat = 'Cash'
    elif vendor == 'Salary':
        cat = 'Salary'

    return vendor, cat

df[['Vendor', 'Category']] = df['Description'].apply(lambda d: pd.Series(extract_vendor_and_category(d)))

# Add DateTime components
df['Month'] = df['Date'].dt.strftime('%b')
df['Month_Num'] = df['Date'].dt.month
df['Hour'] = pd.to_datetime(df['Time'], format='%H:%M').dt.hour

# Function for ASCII bars
def draw_bar(val, max_val, max_chars=15):
    if max_val == 0: return ""
    num_chars = int(round((val / max_val) * max_chars))
    return "#" * num_chars

# Data Slices
credits = df[df['Type'] == 'Credit']
debits = df[df['Type'] == 'Debit'].copy()

# ==============================================================================
# 3. METRICS & ANALYSIS
# ==============================================================================
total_credits = credits['Amount'].sum()
total_debits = debits['Amount'].sum()
net_change = total_credits - total_debits
savings_rate = (net_change / total_credits) * 100 if total_credits > 0 else 0
total_txns = len(df)
unique_vendors = df['Vendor'].nunique()

# Top Categories & Vendors
cat_totals = debits.groupby('Category')['Amount'].sum().sort_values(ascending=False)
top_5_cats = cat_totals.head(5)

vendor_totals = debits.groupby('Vendor').agg(Total_Amount=('Amount', 'sum'), Orders=('Amount', 'count'))
top_vendors = vendor_totals.sort_values(by='Total_Amount', ascending=False).head(5)

# Monthly Trend (Food Delivery)
food_txns = debits[debits['Category'] == 'Food Delivery']
monthly_food = food_txns.groupby(['Month_Num', 'Month'])['Amount'].sum().reset_index().sort_values('Month_Num')

# Hand-Calculated Z-Scores (No SciPy)
cat_stats = debits.groupby('Category')['Amount'].agg(['mean', 'std']).reset_index()
debits_z = debits.merge(cat_stats, on='Category')
debits_z['z_score'] = np.where(debits_z['std'] > 0, (debits_z['Amount'] - debits_z['mean']) / debits_z['std'], 0)
anomalies = debits_z[debits_z['z_score'] >= 3.0].sort_values(by='Date')

# Archetype Ratios
food_pct = (cat_totals.get('Food Delivery', 0) / total_debits) * 100
qcom_pct = (cat_totals.get('Quick Commerce', 0) / total_debits) * 100
ecom_pct = (cat_totals.get('E-commerce', 0) / total_debits) * 100
invest_pct = (cat_totals.get('Investments', 0) / total_debits) * 100
late_night_food = food_txns[food_txns['Hour'] >= 21]['Amount'].sum()
late_night_food_pct = (late_night_food / food_txns['Amount'].sum()) * 100 if food_txns['Amount'].sum() > 0 else 0

# Feature 6 Extension: Day-of-Week Analysis
debits['IsWeekend'] = debits['Date'].dt.dayofweek.isin([5, 6])
weekend_days = debits[debits['IsWeekend']]['Date'].nunique()
weekday_days = debits[~debits['IsWeekend']]['Date'].nunique()

weekend_daily_avg = debits[debits['IsWeekend']]['Amount'].sum() / weekend_days if weekend_days > 0 else 0
weekday_daily_avg = debits[~debits['IsWeekend']]['Amount'].sum() / weekday_days if weekday_days > 0 else 0
weekend_diff_pct = ((weekend_daily_avg - weekday_daily_avg) / weekday_daily_avg) * 100

# Spend Forecasting: NumPy-only 3-Month Rolling Average
monthly_cat_matrix = debits.groupby([debits['Date'].dt.to_period('M'), 'Category'])['Amount'].sum().unstack(fill_value=0)
spend_array = monthly_cat_matrix.to_numpy()
categories = monthly_cat_matrix.columns.to_list()
next_month_forecast = np.mean(spend_array[-3:, :], axis=0) if spend_array.shape[0] >= 3 else np.array([])

# Vendor Cleanup Audit
uncategorised_txns = df[df['Vendor'] == 'Uncategorised']['Description'].unique()

# ==============================================================================
# 4. REPORT DISPLAY
# ==============================================================================
print("====================================================================")
print("  SpendDNA REPORT  -  RAHUL SHARMA")
print(f"  6 months  -  {total_txns:,} transactions  -  Jan to Jun 2024")
print("====================================================================\n")

print("EXECUTIVE SUMMARY")
print(f"  Total credits    :  Rs. {total_credits:,.0f}")
print(f"  Total debits     :  Rs. {total_debits:,.0f}")
print(f"  Net change       : -Rs. {abs(net_change):,.0f}    (overspending)")
print(f"  Savings rate     :  {savings_rate:.1f}%          (BURNING SAVINGS)")
print(f"  Transactions     :  {total_txns:,}")
print(f"  Unique vendors   :  {unique_vendors}\n")

print("TOP CATEGORIES (% of debit total)")
max_cat_val = top_5_cats.max()
for cat, val in top_5_cats.items():
    pct = (val / total_debits) * 100
    bar = draw_bar(val, max_cat_val, 15)
    print(f"  {cat:<15} {bar:<16} {pct:>5.1f}%   Rs. {val:,.0f}")
print()

print("TOP VENDORS")
for vendor, row in top_vendors.iterrows():
    print(f"  {vendor:<15} Rs. {row['Total_Amount']:>8,.0f}  ({row['Orders']:>3} txns)")
print()

print("TIME-OF-DAY & DAY-OF-WEEK PATTERNS")
print(f"  Food Delivery peaks: 21:00 - 01:00  ({late_night_food_pct:.0f}% of orders)")
print(f"  Weekday Daily Avg  : Rs. {weekday_daily_avg:,.0f}/day")
print(f"  Weekend Daily Avg  : Rs. {weekend_daily_avg:,.0f}/day ({abs(weekend_diff_pct):.1f}% {'higher' if weekend_diff_pct > 0 else 'lower'} than weekdays)\n")

print("MONTHLY TREND (Food Delivery)")
max_food_val = monthly_food['Amount'].max()
for _, row in monthly_food.iterrows():
    bar = draw_bar(row['Amount'], max_food_val, 15)
    print(f"  {row['Month']:<4} Rs. {row['Amount']:>6,.0f}  {bar}")
print()

print("TOP ANOMALIES (3+ stddev from category mean)")
for _, row in anomalies.head(2).iterrows():
    date_str = row['Date'].strftime('%d %b')
    print(f"  {date_str:<6} - {row['Vendor']:<10}   Rs. {row['Amount']:>6,.0f}  (z={row['z_score']:.1f})")
print("\n--------------------------------------------------------------------\n")

print("RAHUL'S SPENDING ARCHETYPES")
print(f"  -> THE FOODIE            ({food_pct:.1f}% on food)")
print(f"  -> THE QUICK COMMERCE    ({qcom_pct:.1f}% on Q-com)")
print(f"  -> THE SHOPAHOLIC        ({ecom_pct:.1f}% on e-commerce)")
print(f"  -> THE INVESTOR          ({invest_pct:.1f}% on SIPs)")
print(f"  -> THE LATE-NIGHT SNACKER ({late_night_food_pct:.0f}% food after 9 PM)")
print(f"  -> THE YOLO SPENDER      (savings rate {savings_rate:.0f}%)")
print(f"  -> TECHIE WEEKEND INDULGER (New Rule: Weekend Daily Spend > 1.5x Weekday Daily Spend)\n")

print("NEXT MONTH SPEND FORECAST (NumPy 3-Month Rolling Average)")
for cat, proj in zip(categories, next_month_forecast):
    print(f"  {cat:<20}: Rs. {proj:>8,.0f}")
print()

print("VENDOR CLEANUP AUDIT")
print(f"  Uncategorised Vendors Remaining: {len(uncategorised_txns)} cases")
if len(uncategorised_txns) > 0:
    for desc in uncategorised_txns:
        print(f"   - {desc}")
else:
    print("   - All vendor descriptions successfully mapped (0 uncategorised remaining)!")
print("\n====================================================================")

  SpendDNA REPORT  -  RAHUL SHARMA
  6 months  -  1,328 transactions  -  Jan to Jun 2024

EXECUTIVE SUMMARY
  Total credits    :  Rs. 509,774
  Total debits     :  Rs. 1,729,708
  Net change       : -Rs. 1,219,934    (overspending)
  Savings rate     :  -239.3%          (BURNING SAVINGS)
  Transactions     :  1,328
  Unique vendors   :  22

TOP CATEGORIES (% of debit total)
  E-commerce      ###############   36.9%   Rs. 638,210
  Investments     ######            14.3%   Rs. 248,160
  Food Delivery   ####               9.5%   Rs. 163,812
  Bills & Rent    ####               9.4%   Rs. 162,185
  Commute & Fuel  ####               8.9%   Rs. 154,111

TOP VENDORS
  Amazon          Rs.  348,447  (87.0 txns)
  Zerodha         Rs.  210,000  (14.0 txns)
  Flipkart        Rs.  191,926  (48.0 txns)
  Restaurants     Rs.  117,737  (73.0 txns)
  Rent / Housing  Rs.  108,000  (6.0 txns)

TIME-OF-DAY & DAY-OF-WEEK PATTERNS
  Food Delivery peaks: 21:00 - 01:00  (17% of orders)
  Weekday Daily Avg  